# Week 3a.1: Short-term Memory

Our agents so far have amnesia. Every `invoke` starts from nothing: the model does not remember what you asked in the previous message. 

Today we will address this problem for a single conversation.
- This doesn't apply to memory across conversations and sessions

In [1]:
from dotenv import load_dotenv
import os
import logging

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."

# silence a noisy advisory warning from the Google SDK
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

print("API key loaded")

API key loaded


## 0. Model & Tool Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [2]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [3]:
import requests
from langchain.tools import tool
from langchain_tavily import TavilySearch
from typing import Dict, Any
from datetime import datetime

# WEB SEARCH
@tool
def search_the_web(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    tavily = TavilySearch(max_results=3)
    query_dict = {"query": query}
    results = tavily.invoke(query_dict)
    return results

# CURRENT TIME
@tool
def get_current_time() -> str:
    """Return the current local date and time."""
    return datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")

# WEATHER
@tool
def get_weather(city: str) -> str:
    """Get the weather data for the city provided as an argument"""

    data = requests.get(f"https://wttr.in/{city}?format=j1").json()
    return data["current_condition"][0]

## 1. The problem

In [4]:
response = model.invoke("Hello from Boston.")
print(response.text)

Hello! It's nice to meet you in the City of Trees. How can I help you today?


In [5]:
response = model.invoke("Where am I located?")
print(response.text)

As an AI, I don't have access to your real-time location unless you provide it to me.

To find out where **you** are located, please try one of the following:
*   Check a **GPS app** on your phone or computer.
*   Look at **satellite imagery** (like Google Maps) if you're using an online map.
*   Ask a local resident or check official government resources.
*   Note that for privacy reasons, sharing exact location data with AI assistants should be done cautiously unless it's relevant to the task at hand.

How can I assist you with your current location once you have the information?


The model has no idea. It is **stateless**: nothing carries over from one call to the next. What looked like a conversation in a chat app was never memory inside the model; the application was resending the history every time.

We already have the tool for this: a call can take a **list of messages**. That list is the memory.

That is all short-term memory is: the application rereads the entire conversation to the model on every single call. Nothing is stored inside the model.

## 2. Agents with Threads

For agents, LangChain provides a built-in method to save previous messages into the agent's state (or memory).
- https://docs.langchain.com/oss/python/langchain/short-term-memory

We need to add a **checkpointer** (InMemorySaver) when invoking the `create_agent` method, which will save the conversation state after every call, filed under a `thread_id` we choose. The `thread_id` is typically the unique conversation ID.

Messages with the same thread are saved to the same memory

Add this to agent **declaration**: 
- `checkpointer = InMemorySaver()`

The thread_id is passed as a configurable to agent **invocation**:
- `config = {"configurable": {"thread_id": "1"}}`


In [9]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[get_weather, search_the_web, get_current_time],
    system_prompt="You're a helpful assistant who answers users' questions concisely.",
    checkpointer=InMemorySaver()
)


In [10]:
#TODO define the config

config = {"configurable": {"thread_id":"1"}}

# then add this after the messages dictionary in your agent invocation

In [11]:
from langchain.messages import HumanMessage

message = [HumanMessage(content="Hello from Boston!")]

result = agent.invoke({"messages": message}, config)
print(result["messages"][-1].text)

It's overcast and chilly in Boston right now! The temperature is 59°F (15°C), which feels like about 50°F. There's no precipitation currently, but it looks cloudy with a humidity of 60%. Have a nice evening!


In [12]:
message2 = [HumanMessage(content="Where am I located?")]
result = agent.invoke({"messages": message2}, config)
print(result["messages"][-1].text)

Based on your earlier message, you are in **Boston**. It's currently Tuesday, September 22nd, 2026 at 4:59 PM there.


In [12]:
result

{'messages': [HumanMessage(content='Hello from Boston!', additional_kwargs={}, response_metadata={}, id='206ef961-bf3f-46e6-9823-1af9a1d91174'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-22T20:47:01.551902Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3992856750, 'load_duration': 1820601625, 'prompt_eval_count': 394, 'prompt_eval_duration': 1302919000, 'eval_count': 26, 'eval_duration': 861608000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a0cadf-280d-7b03-a69c-b0e317d0bfd7-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': '9195c01b-c547-4511-8a27-5271ee534ac9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 394, 'output_tokens': 26, 'total_tokens': 420}),
  ToolMessage(content='{"FeelsLikeC": "10", "FeelsLikeF": "50", "cloudcover": "97", "humidity": "60", "observation_time": "08:42 PM", "precipInches": "0.

In [6]:
message3 = [HumanMessage(content="What's the weather like?")]

result = agent.invoke({"messages": message3}, config)
print(result["messages"][-1].text)

NameError: name 'HumanMessage' is not defined

In [14]:
result

{'messages': [HumanMessage(content='Hello from Boston!', additional_kwargs={}, response_metadata={}, id='206ef961-bf3f-46e6-9823-1af9a1d91174'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-22T20:47:01.551902Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3992856750, 'load_duration': 1820601625, 'prompt_eval_count': 394, 'prompt_eval_duration': 1302919000, 'eval_count': 26, 'eval_duration': 861608000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a0cadf-280d-7b03-a69c-b0e317d0bfd7-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': '9195c01b-c547-4511-8a27-5271ee534ac9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 394, 'output_tokens': 26, 'total_tokens': 420}),
  ToolMessage(content='{"FeelsLikeC": "10", "FeelsLikeF": "50", "cloudcover": "97", "humidity": "60", "observation_time": "08:42 PM", "precipInches": "0.

Let's now pass in a different thread:

In [16]:
message2

[HumanMessage(content='Where am I located?', additional_kwargs={}, response_metadata={}, id='d1702e12-75de-4077-b102-f2e83ea428c4')]

In [20]:
config = {"configurable": {"thread_id": "2"}}
result = agent.invoke({"messages": message2}, config)
print(result["messages"][-1].text)

I can't determine your physical location directly as I don't have access to your device's GPS data. However, based on the weather information we looked up, you are currently in **Boston, Massachusetts**.

To find out exactly where you are located on a map yourself, you can:

1.  Use the browser feature (right-click and select "What is this location?")
2.  Ask Siri ("Where am I?") or Alexa ("What's my current location?")
3.  Check the GPS icon in your mobile device status bar for latitude/longitude


## 3. A chat loop

 Let's look at how the chat loop works. Type `quit` to stop.

In [21]:
config = {"configurable": {"thread_id": "live-chat"}}

while True:
    user = input("You: ")

    if user.lower() in {"quit", "exit"}:
        break

    # TODO: turn user input into a human message
    message = [HumanMessage(content=user)]
    # TODO: call the agent with the human message and config
    result = agent.invoke({"messages": message}, config)

    
    print("Assistant:", result["messages"][-1].text)

Assistant: Hello! How can I help you? Do you need weather information for today in Chicago or perhaps general information on a specific topic?
Assistant: It's currently 8:35 PM in Chicago with an overcast sky. The temperature is 60°F (16°C), and it feels like 52°F (52°C) due to the wind. There's a gentle NE breeze at 13 mph, and there are no precipitation plans for the time being. Humidity stands at 63% with a clear visibility of 6 miles.


## 4. A chat window

The terminal loop above works, but one import gives us a real chat interface. `agentui.py` lives next to this notebook: it wraps any agent built with `create_agent`, shows tool calls as collapsible entries while the agent works, and sends the same `thread_id` on every turn, so the agent's own checkpointer does the remembering.

Run the cell and open the local URL it prints. Interrupt the kernel (or restart it) to stop the server.

In [22]:
from agentui import GradioUI

app = GradioUI(agent, {"configurable": {"thread_id": "gradio-demo"}}, title="Agent with Memory")
app.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 5. Tracing

[LangSmith](https://smith.langchain.com/) is LangChain's official tool for tracing agent calls and executions:

To set it up, get an API key from https://smith.langchain.com/ and add it your .env file.
- Set `LANGSMITH_TRACING=true`